In [35]:
# local
from var_check import name_check
from var_check import time_check
from var_check import url_check
from sqlSrcipts import sql_connect
from sqlSrcipts import sql_engine 
# standard
import os
# third
import pandas as pd
import matplotlib.pyplot
import matplotlib.dates as mdates
import requests 

In [ ]:
name = 'Gold Coin Chest'

start = "2026-01-1" #UTC Time
end = "2026-01-13"

condense = 'true'
has_sold = 'true'

limit = '50'
page = '1'

# name_check(name)
# time_check(start, end)

def url():
    url = (f'https://api.darkerdb.com/v1/market'
           f'?key={os.getenv("dark_api_key")}'
           f'&item={name}'
           f'&limit={limit}&page={page}'
           f'&condense={condense}&has_sold={has_sold}'
           f'&from={start}&to={end}')
    return url
# req = requests.get(url())

In [48]:
conn = sql_connect()
cursor = conn.cursor()

type_map = {
    "int64": "BIGINT",
    "float64": "DOUBLE PRECISION",
    "object": "TEXT",
    "datetime64[ns]": "TIMESTAMP",
    "bool" : "BOOLEAN"    
}
df = pd.DataFrame(body_list)
schema = {col: type_map[str(dtype)] for col, dtype in zip(df.columns, df.dtypes)}
columns_dtype = ", ".join(f"{col} {dtype}" for col, dtype in schema.items())
query = f'''
    CREATE TABLE IF NOT EXISTS test (
    {columns_dtype},
    PRIMARY KEY (cursor)
    );'''
cursor.execute(query)


sqlCol = ', '.join(col)
sqlPlaceholder = ", ".join(["%s"] * len(df.columns))
query = f'''
    INSERT INTO test ({sqlCol}) 
    VALUES ({sqlPlaceholder})
    ON CONFLICT (cursor) DO NOTHING;
    '''
rows = [tuple(instance[c] for c in col)for instance in body_list]
cursor.executemany(query,rows)


conn.commit()
conn.close()